In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score, RandomizedSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, classification_report, confusion_matrix,precision_recall_curve
)

from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")
# Gunakan kode dengan hati-hati.

In [6]:
df = pd.read_csv('../data/cleaned_loan_data')

df['term'] = df['term'].astype('str')
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)


df['home_ownership'] = df['home_ownership'].replace(
    {'NONE': 'OTHER', 'ANY': 'OTHER'}
)

nominal_features   = ['home_ownership', 'verification_status', 'initial_list_status']
ordinal_features   = ['grade']
ordinal_categories = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]

num_features = [c for c in df.select_dtypes(include=['int64', 'float64']).columns
                if c != 'target'
                and c not in ordinal_features
                and c not in nominal_features]

target = 'target'

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# print(X_train.shape,y_train.shape,X_test.shape,y_test.shape)

RANDOM_STATE = 42
N_ITER= 3
CV_FOLDS= 2
 
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
cv= StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [7]:
numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Ordinal
ordinal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Nominal
nominal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,  num_features),
    ('ord', ordinal_pipeline,  ordinal_features),
    ('nom', nominal_pipeline,  nominal_features),
], remainder='drop')

In [8]:
lr_param_dist = {
    'model__C':        [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
    'model__penalty':  ['l1', 'l2'],
    'model__l1_ratio':  [0.0, 0.25, 0.5, 0.75, 1.0],
    'model__max_iter': [500, 1000, 1500, 2000],
}
 
lr_base = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE, k_neighbors=5)),
    ('model', LogisticRegression(
        solver='saga',
        class_weight='balanced',
        random_state=RANDOM_STATE,
#         n_jobs=1
    ))
])
 
lr_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=N_ITER,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

lr_search.fit(X_train, y_train)

print(f"  Best AUC-ROC (CV) : {lr_search.best_score_:.4f}")
print(f"  Best Params:")
for k, v in lr_search.best_params_.items():
    print(f"    {k:<25} : {v}")

Fitting 2 folds for each of 3 candidates, totalling 6 fits



KeyboardInterrupt


KeyboardInterrupt

